# 01. Python Fundamentals

## What you'll learn

- **Variables, types, and operators** — the building blocks of every Python program
- **Strings and f-strings** — formatting text, which is how we construct prompts for LLMs
- **Control flow** — `if`/`elif`/`else` for making decisions (like routing agent actions)
- **Loops** — `for` and `while` loops, including `break` (the core of an agent loop)
- **Imports and modules** — reusing code from the standard library and third-party packages
- **Jupyter basics** — magic commands and shell integration for interactive development

## Prerequisites

None. This is the starting point.

## Why this matters for agents

An AI agent is, at its core, a **loop**: receive input, call an LLM, parse the response, decide what to do next, repeat. Every concept in this notebook — variables to hold state, strings to build prompts, conditionals to route decisions, loops to keep running — maps directly onto the agent primitives you'll build in the `core/` track.

> **Further reading:**
> - [The Python Tutorial (official docs)](https://docs.python.org/3/tutorial/) — the authoritative introduction, chapters 3-5 cover everything here in depth
> - [Automate the Boring Stuff, Chapters 1-2](https://automatetheboringstuff.com/) — a gentler, project-oriented walkthrough of fundamentals
> - [Think Python 3e (free online book)](https://greenteapress.com/wp/think-python-3rd-edition/) — complete intro, CC-licensed
> - [Python for Everybody (free book + Coursera)](https://www.py4e.com/) — beginner-friendly, video + text
> - [Corey Schafer — Python Tutorials (YouTube playlist)](https://www.youtube.com/playlist?list=PL-osiE80TeTt2d9bfVyTiXJA-UTHn6WwU) — excellent video series covering basics
> - [Kaggle — Intro to Python (interactive)](https://www.kaggle.com/learn/python) — free, browser-based exercises
> - [Python Tutor (visualizer)](https://pythontutor.com/) — step through code visually, see variables change
> - [CS50P — Introduction to Programming with Python (Harvard)](https://cs50.harvard.edu/python/) — full course, free

---
## 1. Variables, Types, and Operators

Variables are names that point to values. Python figures out the type automatically — you don't need to declare it. When building agents, variables hold things like the current prompt, a step counter, or a list of messages.

In [ ]:
# Basic variable assignments — imagine these as agent configuration
model_name = "openrouter/auto"       # str
max_steps = 10                        # int
temperature = 0.7                     # float
verbose = True                        # bool

print(type(model_name))   # <class 'str'>
print(type(max_steps))    # <class 'int'>
print(type(temperature))  # <class 'float'>
print(type(verbose))      # <class 'bool'>

In [ ]:
# Arithmetic operators
tokens_used = 150
tokens_limit = 4096
tokens_remaining = tokens_limit - tokens_used

print(f"Remaining tokens: {tokens_remaining}")
print(f"Usage percentage: {tokens_used / tokens_limit * 100:.1f}%")

# Integer division and modulo — useful for batching
total_items = 17
batch_size = 5
full_batches = total_items // batch_size
leftover = total_items % batch_size
print(f"{full_batches} full batches, {leftover} remaining")

In [ ]:
# Comparison and logical operators — used constantly in agent decisions
step = 3
is_within_limit = step < max_steps           # True
is_done = step >= max_steps                   # False
should_continue = is_within_limit and verbose # True

print(f"Step {step}: within limit? {is_within_limit}, done? {is_done}")
print(f"Should continue? {should_continue}")

---
## 2. Strings and f-strings

Strings are the primary data type you work with in LLM applications. Prompts are strings. Responses are strings. Tool definitions are strings. Mastering string formatting is essential.

In [ ]:
# f-strings let you embed expressions directly in a string
user_query = "What's the weather in Tokyo?"
role = "assistant"

prompt = f"You are a helpful {role}. The user asked: {user_query}"
print(prompt)

# Expressions inside the braces
step = 3
print(f"Step {step}/{max_steps} ({step / max_steps * 100:.0f}% complete)")

In [ ]:
# Triple-quoted strings — perfect for multi-line prompts
system_prompt = """You are a research assistant agent.

Your capabilities:
- Search the web for information
- Summarize long documents
- Answer follow-up questions

Always cite your sources."""

print(system_prompt)

In [ ]:
# Useful string methods for parsing LLM output
llm_response = "  ACTION: search('Python tutorials')  \n"

cleaned = llm_response.strip()          # Remove surrounding whitespace
print(f"Stripped: '{cleaned}'")

starts_with_action = cleaned.startswith("ACTION:")
print(f"Is an action? {starts_with_action}")

action_body = cleaned.replace("ACTION: ", "")
print(f"Action body: '{action_body}'")

# Split is great for tokenizing simple formats
parts = "user|What is 2+2?|high".split("|")
print(f"Role: {parts[0]}, Content: {parts[1]}, Priority: {parts[2]}")

---
## 3. Control Flow

Agents make decisions constantly: Should I call a tool or respond directly? Is the task complete? Did the LLM return an error? All of this is `if`/`elif`/`else`.

In [ ]:
# Simple agent action router
action = "search"

if action == "search":
    print("Calling search tool...")
elif action == "calculate":
    print("Calling calculator tool...")
elif action == "respond":
    print("Generating final response...")
else:
    print(f"Unknown action: {action}")

In [ ]:
# Nested conditions — checking multiple things before proceeding
step = 5
max_steps = 10
has_error = False
task_complete = False

if step >= max_steps:
    status = "stopped: max steps reached"
elif has_error:
    status = "stopped: error encountered"
elif task_complete:
    status = "completed successfully"
else:
    status = "running"

print(f"Agent status at step {step}: {status}")

In [ ]:
# Truthiness — Python treats empty/zero values as False
# This comes up often when checking if an LLM returned anything
response_text = ""
tool_calls = []
retry_count = 0

if not response_text:
    print("Empty response from LLM")

if not tool_calls:
    print("No tool calls requested")

if not retry_count:
    print("No retries yet (0 is falsy)")

# Contrast with non-empty values
response_text = "Hello!"
if response_text:
    print(f"Got a response: {response_text}")

---
## 4. Loops

The agent loop is literally a loop. You'll use `for` loops to iterate over messages, tool results, and batches. You'll use `while` loops for the main agent execution cycle — running until the task is done or a limit is hit.

In [ ]:
# for loop — iterate over a list of messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a programming language."},
]

for message in messages:
    role = message["role"]
    content = message["content"]
    print(f"[{role.upper()}] {content}")

In [ ]:
# range() and enumerate() — two essential patterns
tools = ["search", "calculator", "weather", "calendar"]

# range gives you a sequence of numbers
print("Available tool indices:")
for i in range(len(tools)):
    print(f"  [{i}] {tools[i]}")

print()

# enumerate is the Pythonic way — gives you both index and value
print("Available tools:")
for idx, tool_name in enumerate(tools):
    print(f"  [{idx}] {tool_name}")

In [ ]:
# while loop with break — the fundamental agent loop pattern
max_steps = 5
step = 0

while step < max_steps:
    step += 1
    print(f"Step {step}: thinking...")

    # Simulate a condition where the agent decides it's done
    if step == 3:
        print(f"Step {step}: task complete! Breaking out of loop.")
        break

print(f"\nLoop ended after {step} steps.")

In [ ]:
# continue — skip an iteration without stopping the loop
actions = ["search", "ERROR", "calculate", "ERROR", "respond"]

for action in actions:
    if action == "ERROR":
        print(f"  Skipping invalid action: {action}")
        continue
    print(f"  Executing: {action}")

---
## 5. Imports and Modules

Python's power comes from its ecosystem. You'll import standard library modules (`json`, `os`, `random`) and third-party packages (`httpx`, `dotenv`). Understanding imports is essential for building anything beyond a single cell.

In [ ]:
# Import an entire module
import math

print(f"Pi: {math.pi}")
print(f"sqrt(144): {math.sqrt(144)}")

# Import specific things from a module
from random import choice, randint

actions = ["search", "calculate", "respond"]
random_action = choice(actions)
random_delay = randint(1, 5)
print(f"Random action: {random_action}, delay: {random_delay}s")

In [ ]:
# Modules you'll use constantly in this workshop
import json
import os
import time

# json — LLM APIs send and receive JSON
tool_definition = {
    "name": "search",
    "description": "Search the web for information",
    "parameters": {"query": "string"},
}
json_string = json.dumps(tool_definition, indent=2)
print(json_string)

# Parse it back
parsed = json.loads(json_string)
print(f"\nTool name: {parsed['name']}")

In [ ]:
# os — environment variables (this is how you'll load API keys)
# In a real project you'd use python-dotenv, but os.environ is the foundation
os.environ["DEMO_API_KEY"] = "sk-demo-12345"  # Don't do this with real keys!

api_key = os.environ.get("DEMO_API_KEY", "not-set")
print(f"API key loaded: {api_key[:8]}...")

# .get() with a default is safer than direct access
missing_key = os.environ.get("NONEXISTENT_KEY", "default_value")
print(f"Missing key fallback: {missing_key}")

# Clean up
del os.environ["DEMO_API_KEY"]

---
## 6. Jupyter Basics

Jupyter notebooks are our development environment throughout this workshop. A few built-in features make interactive development much faster.

In [ ]:
# %time — measure how long a single statement takes
# (useful for benchmarking LLM calls later)
import time

%time time.sleep(0.1)  # Should take ~100ms

In [ ]:
# %timeit — run something many times for a precise measurement
# Great for comparing string formatting approaches
name = "agent"
%timeit f"hello {name}"       # f-strings
%timeit "hello " + name       # concatenation
%timeit "hello {}".format(name)  # .format()

In [ ]:
# ? — get help on any object (try this interactively!)
# Uncomment the line below to see the docstring for len:
# len?

In [ ]:
# ! — run shell commands from inside a notebook
# This is how you'll install packages and check your environment
!python --version

A few other things worth knowing about Jupyter:

- The **last expression** in a cell is automatically displayed (no `print()` needed).
- **Shift+Enter** runs the current cell and moves to the next.
- **Tab** after a dot (e.g., `str.`) shows autocomplete options.
- Variables persist across cells within the same session — that's how we build up state incrementally.

---
## Putting It Together: A Simulated Agent Loop

Let's combine everything we've learned into a single, realistic example: a simulated agent loop. This is the pattern you'll implement for real in the `core/` notebooks — a `while` loop that runs until the agent decides it's done or hits a step limit.

In [ ]:
import json
from random import choice

# --- Agent configuration ---
agent_name = "ResearchBot"
max_steps = 6
available_tools = ["search", "summarize", "respond"]

system_prompt = f"""You are {agent_name}, a research assistant.
You have access to: {', '.join(available_tools)}.
Use tools to gather info, then respond."""

print(f"System prompt:\n{system_prompt}\n")
print("=" * 50)

# --- Agent loop ---
step = 0
messages = [{"role": "system", "content": system_prompt}]
task_complete = False

while step < max_steps and not task_complete:
    step += 1

    # Simulate the LLM "choosing" an action
    action = choice(available_tools)
    print(f"\nStep {step}/{max_steps}: LLM chose action -> {action}")

    # Route the action
    if action == "search":
        result = "Found 3 relevant articles about Python agents."
    elif action == "summarize":
        result = "Summary: Agents use loops, tools, and LLM calls."
    elif action == "respond":
        result = "Final answer delivered to the user."
        task_complete = True
    else:
        result = f"Unknown action: {action}"

    # Record the step
    messages.append({"role": "assistant", "content": f"[{action}] {result}"})
    print(f"  Result: {result}")

# --- Summary ---
print("\n" + "=" * 50)
if task_complete:
    print(f"{agent_name} completed in {step} steps.")
else:
    print(f"{agent_name} hit the step limit ({max_steps}).")

print(f"\nFull message log ({len(messages)} messages):")
print(json.dumps(messages, indent=2))

Notice how every concept from this notebook appears in the capstone:

| Concept | Where it shows up |
|---|---|
| Variables & types | `agent_name`, `max_steps`, `task_complete` |
| f-strings | The system prompt, print statements |
| Control flow | `if`/`elif`/`else` action router |
| Loops | `while` loop with `break` condition |
| Imports | `json`, `random.choice` |

This is the skeleton of every agent you'll build.

---
## Try It Yourself

Work through these exercises to reinforce what you've learned. Each one practices a different concept.

### Exercise 1: Countdown Loop

Write a `while` loop that counts down from 5 to 1, printing each number, then prints `"Liftoff!"`. Use an f-string for the output (e.g., `"T-minus 5..."`).

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Build a Prompt from Parts

Given the list of words below, use a `for` loop to join them into a single sentence string. Then wrap it in a system prompt using an f-string and a triple-quoted string.

```python
words = ["You", "are", "a", "helpful", "coding", "assistant"]
```

Expected output should look like:
```
System prompt:
You are a helpful coding assistant.
Always explain your reasoning step by step.
```

In [ ]:
# Exercise 2: Your code here
words = ["You", "are", "a", "helpful", "coding", "assistant"]


### Exercise 3: Agent Step Simulation with Random Early Stop

Write an agent loop that:
1. Runs for up to 8 steps.
2. On each step, uses `random.random()` to generate a number between 0 and 1.
3. If the random number is greater than 0.7, the agent "decides" it's done — print a message and `break`.
4. Otherwise, print the step number and the random value.
5. After the loop, print how many steps were executed.

Hint: `import random` and use `random.random()` for a float in [0, 1).

In [ ]:
# Exercise 3: Your code here
import random


### Exercise 4 (Stretch): Message Formatter

Given the list of message dicts below, write a loop that prints each one in a formatted style. For `"system"` messages, prefix with `[SYSTEM]`. For `"user"` messages, prefix with `[USER]`. For `"assistant"` messages, prefix with `[AGENT]`. If the role is anything else, print `[UNKNOWN]`.

```python
messages = [
    {"role": "system", "content": "You are a travel planner agent."},
    {"role": "user", "content": "Plan a weekend trip to Kyoto."},
    {"role": "assistant", "content": "I'll search for flights and hotels."},
    {"role": "tool", "content": "Found 5 flights, 12 hotels."},
    {"role": "assistant", "content": "Here's your itinerary..."},
]
```

In [ ]:
# Exercise 4: Your code here
messages = [
    {"role": "system", "content": "You are a travel planner agent."},
    {"role": "user", "content": "Plan a weekend trip to Kyoto."},
    {"role": "assistant", "content": "I'll search for flights and hotels."},
    {"role": "tool", "content": "Found 5 flights, 12 hotels."},
    {"role": "assistant", "content": "Here's your itinerary..."},
]


---

**Next up:** [02. Functions and Scope](02_functions_and_scope.ipynb) — defining reusable functions, which is how you'll encapsulate tool implementations and agent logic.